# FactoryTwin AI — Setup & Run
# Open this notebook in VS Code. Use the integrated terminal/output pane.
# Prerequisites: Python 3.8+, git (optional), node/npm (for frontend).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import shutil
import json

PROJECT_ROOT = Path.cwd()
BACKEND_DIR = PROJECT_ROOT / "backend"

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)

# helper
def run(cmd, cwd=None, stream=False):
    '''Run shell command; return (retcode, stdout, stderr)'''
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True, text=True)
    out, err = p.communicate()
    return p.returncode, out, err



# Clone or copy project (optional)
# If you need to clone elsewhere, uncomment and edit the URL below.
# Example (git must be installed):
# ret, out, err = run('git clone https://github.com/your/repo.git target_dir')
# print(ret, out, err)

print('Repository already present at', PROJECT_ROOT)


In [ ]:
# Create virtual environment (optional)
venv_dir = PROJECT_ROOT / '.venv'
if not venv_dir.exists():
    print('Creating venv at', venv_dir)
    subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)])
else:
    print('.venv already exists')

print('\nActivation commands:')
print('  Bash (macOS/Linux): source .venv/bin/activate')
print('  PowerShell (Windows): .\.venv\Scripts\Activate.ps1')
print('  CMD (Windows): .\.venv\Scripts\activate.bat')


In [ ]:
# Install backend dependencies
print('Upgrading pip...')
ret, out, err = run(f'{sys.executable} -m pip install --upgrade pip')
print('pip upgrade ret:', ret)
print(out, err)

req = BACKEND_DIR / 'requirements.txt'
if req.exists():
    print('Installing requirements from', req)
    ret, out, err = run(f'{sys.executable} -m pip install -r "{req}"')
    print('install ret:', ret)
    print(out)
    if err:
        print('install err:', err)
else:
    print('No requirements.txt at', req)


In [ ]:
# Verify sentence-transformers and embedding dimension
from dotenv import load_dotenv
load_dotenv(str(BACKEND_DIR / '.env')) if (BACKEND_DIR / '.env').exists() else load_dotenv()
EMBED = os.getenv('EMBEDDING_MODEL', 'all-MiniLM-L6-v2')
print('EMBEDDING_MODEL=', EMBED)
try:
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(EMBED)
    dim = getattr(m, 'get_embedding_dimension', None) or getattr(m, 'get_sentence_embedding_dimension', None)
    print('embedding dim =', dim())
except Exception as e:
    print('Error loading embedding model:', e)
    print('If this fails, ensure backend dependencies are installed (see previous cell).')


In [ ]:
# Inspect Qdrant collections and vector config
try:
    from qdrant_client import QdrantClient
    c = QdrantClient(path=str(PROJECT_ROOT / 'database' / 'qdrant_data'))
    cols = c.get_collections().collections
    if not cols:
        print('No collections found in database/qdrant_data')
    for col in cols:
        print('Collection:', col.name)
        try:
            info = c.get_collection(col.name)
            print('  vectors:', getattr(info, 'vectors', None))
        except Exception as e:
            print('  could not get full info:', e)
    c.close()
except Exception as e:
    print('qdrant inspect error:', e)
    print('If qdrant-client is not installed or Qdrant files missing, check database/qdrant_data')


In [ ]:
# Optionally delete the 'endpoints' collection (interactive)
confirm = input("Delete 'endpoints' collection and recreate? [y/N]: ")
if confirm.lower() == 'y':
    try:
        from qdrant_client import QdrantClient
        c = QdrantClient(path=str(PROJECT_ROOT / 'database' / 'qdrant_data'))
        c.delete_collection('endpoints')
        c.close()
        print("Deleted 'endpoints' collection")
    except Exception as e:
        print('Error deleting collection:', e)
        print("Alternative (PowerShell): Remove-Item -Recurse -Force .\\database\\qdrant_data")
else:
    print('Skipping delete')


In [ ]:
# Repopulate Qdrant via populate_vector_db.py
print('Running populate_vector_db.py...')
ret, out, err = run(f'{sys.executable} scripts/populate_vector_db.py', cwd=str(BACKEND_DIR))
print('ret=', ret)
print(out)
if err:
    print('err=', err)


In [ ]:
# Start backend (uvicorn) in background and log to file
import subprocess, time
log_file = BACKEND_DIR / 'uvicorn.log'
if log_file.exists():
    print('uvicorn log exists at', log_file)

cmd = [sys.executable, '-m', 'uvicorn', 'api.server:app', '--host', '0.0.0.0', '--port', '8000', '--reload']
print('Starting uvicorn with:', ' '.join(cmd))
# open log file in append mode
f = open(log_file, 'ab')
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.server:app', '--host', '0.0.0.0', '--port', '8000', '--reload'], cwd=str(BACKEND_DIR), stdout=f, stderr=f)
print('uvicorn PID:', proc.pid)
print('To stop: proc.terminate() or kill the PID shown above; logs at', log_file)


# Frontend startup & test instructions

print('Frontend:')
print('  cd frontend')
print('  npm install')
print('  npm run dev')

print('\nQuick API test (after backend is running):')
print("curl -sS -X POST http://localhost:8000/api/chat -H 'Content-Type: application/json' -d '{\"query\":\"Show me total aggregate demand for Minneapolis\",\"conversation_id\":null}'")
